In [9]:
import pandas as pd
import numpy as np

merged_path  = "merged_mycotoxins.csv"
merged = pd.read_csv(merged_path)

C:\Users\homme015\AppData\Local\Temp\ipykernel_40744\2953486749.py:5: DtypeWarning: Columns (0: sample_id, 1: dataprovider, 2: date_analysis, 3: method_analysis, 4: unit, 5: continent, 6: country, 7: province, 8: town, 9: quality, 10: most_detailed_region, 11: governance_index, 12: consumer_prices_index_food, 13: food_price_inflation, 14: oil_price, 15: oil_price_std, 16: gas_price, 17: gas_price_std, 18: import_price_global, 19: export_price_global, 20: month_1_precipitation_mean, 21: month_2_precipitation_mean, 22: month_3_precipitation_mean, 23: month_4_precipitation_mean, 24: month_1_precipitation_mean_std, 25: month_2_precipitation_mean_std, 26: month_3_precipitation_mean_std, 27: month_4_precipitation_mean_std, 28: progtype, 29: labaccred, 30: samppoint, 31: sampmethod, 32: sampstrategy, 33: origcountry, 34: evalcode, 35: restype, 36: samplehash, 37: sampid, 38: sampanid, 39: progid, 40: proglegalref, 41: labcountry, 42: filename, 43: country_code, 44: filetype, 45: resid, 46: pa

In [10]:
print("Shape:", merged.shape)

weather_cols = [c for c in merged.columns if c.startswith("month_") or c.startswith("week_")]
print(f"Weather columns found: {len(weather_cols)}")

sample_counts = merged.groupby("sample_id").size()
multi_ids = sample_counts[sample_counts > 1].index
print(f"Samples with >1 row (multiple contaminants tested): {len(multi_ids)} / {len(sample_counts)}")

multi_data = merged[merged["sample_id"].isin(multi_ids)]




Shape: (12243, 447)
Weather columns found: 140
Samples with >1 row (multiple contaminants tested): 1960 / 2430


In [11]:


# ============================================================
# Paths
# ============================================================

INPUT_PATH  = "merged_mycotoxins.csv"
OUTPUT_PATH = "sample_level_dataset.csv"


# ============================================================
# Load data
# ============================================================

merged = pd.read_csv(INPUT_PATH, low_memory=False)

print(f"Loaded: {merged.shape}")

weather_cols = [
    c for c in merged.columns 
    if c.startswith("month_") or c.startswith("week_")
]

print(f"Weather columns: {len(weather_cols)}")


# ============================================================
# Recalculate LOQ-based detection thresholds AFTER merge
# ============================================================

loq_median = merged["loq"].median()
loq_max = merged["loq"].max()

print("\nLOQ thresholds (post-merge):")
print(f"  Median LOQ : {loq_median}")
print(f"  Max LOQ    : {loq_max}")
print(f"  N with LOQ : {merged['loq'].notna().sum():,}")
print(f"  N missing  : {merged['loq'].isna().sum():,}")


# ============================================================
# Global LOQ floor detection
# ============================================================

val = merged["value"].fillna(0)

# Median LOQ floor
merged["detected2"] = (
    val > loq_median
).astype("int8")


# Maximum LOQ floor
merged["detected3"] = (
    val > loq_max
).astype("int8")


# ============================================================
# Contaminant-specific median LOQ floor detection
# ============================================================

loq_median_contaminant = (
    merged.groupby("contaminant")["loq"]
    .median()
    .rename("loq_median_contaminant")
)

merged = merged.join(
    loq_median_contaminant,
    on="contaminant"
)


# Optional fallback:
# if a contaminant has no LOQ values, use global median LOQ
merged["loq_median_contaminant"] = (
    merged["loq_median_contaminant"]
    .fillna(loq_median)
)


merged["detectedC"] = (
    val > merged["loq_median_contaminant"]
).astype("int8")


print("\nExample contaminant LOQ floors:")
print(
    merged[
        ["contaminant", "loq_median_contaminant"]
    ]
    .drop_duplicates()
    .sort_values("contaminant")
    .head(20)
)


# ============================================================
# Aggregate to sample level
# ============================================================

KEEP_COLS = [
    "sample_id",
    "samplecountry",
    "harvest_year",
    "product",
    "product_harmonised"
] + weather_cols


# first = same for sample
agg_dict = {
    col: "first" 
    for col in KEEP_COLS 
    if col != "sample_id"
}


# any contaminant detected in sample
agg_dict["detected"]  = "max"
agg_dict["detected2"] = "max"
agg_dict["detected3"] = "max"
agg_dict["detectedC"] = "max"


sample_df = (
    merged
    .groupby("sample_id")
    .agg(agg_dict)
    .reset_index()
)


# ============================================================
# Diagnostics
# ============================================================

print(f"\nAggregated shape: {sample_df.shape}")

print(
    f"\nsamplecountry:\n"
    f"{sample_df['samplecountry'].value_counts()}"
)

print(
    f"\nharvest_year:\n"
    f"{sample_df['harvest_year'].value_counts().sort_index()}"
)


print("\nMissingness in weather cols:")
print(
    sample_df[weather_cols]
    .isna()
    .mean()
    .sort_values(ascending=False)
    .head(10)
)


print(
    f"\ndetected (LOD-based):\n"
    f"{sample_df['detected'].value_counts(dropna=False)}"
)

print(
    f"\ndetected2 (median LOQ={loq_median}):\n"
    f"{sample_df['detected2'].value_counts(dropna=False)}"
)

print(
    f"\ndetected3 (max LOQ={loq_max}):\n"
    f"{sample_df['detected3'].value_counts(dropna=False)}"
)

print(
    "\ndetectedC (contaminant median LOQ floor):\n"
    f"{sample_df['detectedC'].value_counts(dropna=False)}"
)


for col in ["detected", "detected2", "detected3", "detectedC"]:
    rate = sample_df[col].mean()
    print(f"  {col} detection rate: {rate:.1%}")


# ============================================================
# Save
# ============================================================

sample_df.to_csv(
    OUTPUT_PATH,
    index=False
)

print(f"\nSaved -> {OUTPUT_PATH}")

Loaded: (12243, 447)
Weather columns: 140

LOQ thresholds (post-merge):
  Median LOQ : 5.0
  Max LOQ    : 100.0
  N with LOQ : 12,243
  N missing  : 0

Example contaminant LOQ floors:
                       contaminant  loq_median_contaminant
5278                     altenuene                    8.00
9                      alternariol                    4.60
5176  alternariol monomethyl ether                    5.00
5512                  altertoxin i                   30.00
6                      beauvericin                   15.00
5184                    enniatin a                   10.00
7                  enniatin a + a1                    0.09
5178                   enniatin a1                   10.00
5171                    enniatin b                   10.00
8                  enniatin b + b1                    0.22
5170                   enniatin b1                   10.00
4                    ergocorninine                    2.76
2                   ergocristinine               

C:\Users\homme015\AppData\Local\Temp\ipykernel_40744\3769039880.py:81: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged["detectedC"] = (
C:\Users\homme015\AppData\Local\Temp\ipykernel_40744\3769039880.py:129: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  .reset_index()



Saved -> sample_level_dataset.csv


In [12]:
# ============================================================
# Unique samples per country per year
# ============================================================

sample_country_year = (
    sample_df
    .groupby(["samplecountry", "harvest_year"])["sample_id"]
    .nunique()
    .unstack(fill_value=0)
    .sort_index(axis=1)
)

# Add totals
sample_country_year["Total"] = sample_country_year.sum(axis=1)

total_row = sample_country_year.sum(axis=0)
total_row.name = "Total"

sample_country_year = pd.concat(
    [sample_country_year, total_row.to_frame().T]
)

print("\nUnique samples per country per year:")
print(sample_country_year)

sample_country_year.to_csv("unique_samples_per_country_year.csv")

print("\nSaved -> unique_samples_per_country_year.csv")


Unique samples per country per year:
harvest_year  2019.0  2020.0  2021.0  2022.0  2023.0  Total
cz                26      34      54      72      62    248
de                 4      63      70      58      29    224
dk               143     149     167     208     173    840
nl                83     122      91      88      40    424
Total            256     368     382     426     304   1736

Saved -> unique_samples_per_country_year.csv


In [13]:
# ============================================================
# Detection summary per contaminant
# ============================================================

# Original detection (using each record's own LOQ)
merged["detected_original"] = (
    merged["value"].fillna(0) > merged["loq"]
).astype("int8")

contaminant_summary = (
    merged.groupby("contaminant")
    .agg(
        n_records=("contaminant", "size"),
        median_loq=("loq_median_contaminant", "first"),
        detected_original=("detected_original", "sum"),
        detectedC=("detectedC", "sum")
    )
    .reset_index()
)

contaminant_summary["original_detection_rate"] = (
    100 * contaminant_summary["detected_original"]
    / contaminant_summary["n_records"]
)

contaminant_summary["detectedC_rate"] = (
    100 * contaminant_summary["detectedC"]
    / contaminant_summary["n_records"]
)

contaminant_summary["detections_lost"] = (
    contaminant_summary["detected_original"]
    - contaminant_summary["detectedC"]
)

contaminant_summary["loss_pct_of_detected"] = (
    100
    * contaminant_summary["detections_lost"]
    / contaminant_summary["detected_original"].clip(lower=1)
)

print("\nDetection summary per contaminant")
print(
    contaminant_summary.round(2).to_string(index=False)
)

contaminant_summary.to_csv(
    "detectedC_summary_per_contaminant.csv",
    index=False
)


Detection summary per contaminant
                 contaminant  n_records  median_loq  detected_original  detectedC  original_detection_rate  detectedC_rate  detections_lost  loss_pct_of_detected
                   altenuene        125        8.00                  0          0                     0.00            0.00                0                  0.00
                 alternariol        993        4.60                 35         35                     3.52            3.52                0                  0.00
alternariol monomethyl ether        442        5.00                 13         13                     2.94            2.94                0                  0.00
                altertoxin i         16       30.00                  0          0                     0.00            0.00                0                  0.00
                 beauvericin       1676       15.00                174        123                    10.38            7.34               51                

C:\Users\homme015\AppData\Local\Temp\ipykernel_40744\404383701.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged["detected_original"] = (


In [15]:
# ============================================================
# Unique samples per crop per year
# ============================================================

sample_crop_year = (
    sample_df
    .groupby(["product_harmonised", "harvest_year"])["sample_id"]
    .nunique()
    .unstack(fill_value=0)
    .sort_index(axis=1)
)

sample_crop_year["Total"] = sample_crop_year.sum(axis=1)
total_row = sample_crop_year.sum(axis=0)
total_row.name = "Total"
sample_crop_year = pd.concat([sample_crop_year, total_row.to_frame().T])

print("\nUnique samples per crop per year:")
print(sample_crop_year)

sample_crop_year.to_csv("unique_samples_per_crop_year.csv")
print("\nSaved -> unique_samples_per_crop_year.csv")


# ============================================================
# Unique samples per crop per country per year
# ============================================================

sample_crop_country_year = (
    sample_df
    .groupby(["product_harmonised", "samplecountry", "harvest_year"])["sample_id"]
    .nunique()
    .unstack(fill_value=0)
    .sort_index(axis=1)
)

sample_crop_country_year["Total"] = sample_crop_country_year.sum(axis=1)

print("\nUnique samples per crop per country per year:")
print(sample_crop_country_year)

sample_crop_country_year.to_csv("unique_samples_per_crop_country_year.csv")
print("\nSaved -> unique_samples_per_crop_country_year.csv")


# ============================================================
# Fungal metabolite prevalence per crop
# ============================================================
# For each crop, how many samples were tested for each contaminant, and
# what fraction were detected (using the same "detectedC" flag as above).

merged["detected_original"] = (
    merged["value"].fillna(0) > merged["loq"]
).astype("int8")

prevalence_per_crop = (
    merged.groupby(["product_harmonised", "contaminant"])
    .agg(
        n_samples_tested=("sample_id", "nunique"),
        n_detected=("detectedC", "sum"),
    )
    .reset_index()
)

prevalence_per_crop["detection_rate_pct"] = (
    100 * prevalence_per_crop["n_detected"] / prevalence_per_crop["n_samples_tested"]
)

print("\nFungal metabolite prevalence per crop:")
print(prevalence_per_crop.round(2).to_string(index=False))

prevalence_per_crop.to_csv("metabolite_prevalence_per_crop.csv", index=False)
print("\nSaved -> metabolite_prevalence_per_crop.csv")

# Pivoted version: crop x contaminant, detection rate, easier to scan / paste into thesis
prevalence_pivot = prevalence_per_crop.pivot(
    index="contaminant", columns="product_harmonised", values="detection_rate_pct"
).round(1)
prevalence_pivot.to_csv("metabolite_prevalence_per_crop_pivot.csv")
print("\nSaved -> metabolite_prevalence_per_crop_pivot.csv")




Unique samples per crop per year:
harvest_year  2019.0  2020.0  2021.0  2022.0  2023.0  Total
barley            92     124     153     199     126    694
maize             61      86      50      71      19    287
wheat            103     158     179     156     159    755
Total            256     368     382     426     304   1736

Saved -> unique_samples_per_crop_year.csv

Unique samples per crop per country per year:
harvest_year                      2019.0  2020.0  2021.0  2022.0  2023.0  \
product_harmonised samplecountry                                           
barley             cz                 11       7      17      33      25   
                   de                  2      22      41      29       3   
                   dk                 60      70      79     107      82   
                   nl                 19      25      16      30      16   
maize              cz                  4       4       7      22       3   
                   de                  1   

In [16]:
# ============================================================
# Detection rate (detectedC) summary tables
# ============================================================

def detection_rate_table(df, group_cols):
    t = (
        df.groupby(group_cols)["detectedC"]
        .agg(n_samples="count", n_detected="sum")
        .reset_index()
    )
    t["detection_rate_pct"] = 100 * t["n_detected"] / t["n_samples"]
    return t.round(2)

# Per crop
rate_by_crop = detection_rate_table(sample_df, ["product_harmonised"])
print("\nDetection rate per crop:")
print(rate_by_crop.to_string(index=False))
rate_by_crop.to_csv("detection_rate_per_crop.csv", index=False)

# Per year
rate_by_year = detection_rate_table(sample_df, ["harvest_year"])
print("\nDetection rate per year:")
print(rate_by_year.to_string(index=False))
rate_by_year.to_csv("detection_rate_per_year.csv", index=False)

# Per country
rate_by_country = detection_rate_table(sample_df, ["samplecountry"])
print("\nDetection rate per country:")
print(rate_by_country.to_string(index=False))
rate_by_country.to_csv("detection_rate_per_country.csv", index=False)

# Per crop x year
rate_by_crop_year = detection_rate_table(sample_df, ["product_harmonised", "harvest_year"])
pivot_crop_year = rate_by_crop_year.pivot(
    index="product_harmonised", columns="harvest_year", values="detection_rate_pct"
)
print("\nDetection rate per crop x year (%):")
print(pivot_crop_year.round(1))
pivot_crop_year.to_csv("detection_rate_per_crop_year.csv")

# Per crop x country
rate_by_crop_country = detection_rate_table(sample_df, ["product_harmonised", "samplecountry"])
pivot_crop_country = rate_by_crop_country.pivot(
    index="product_harmonised", columns="samplecountry", values="detection_rate_pct"
)
print("\nDetection rate per crop x country (%):")
print(pivot_crop_country.round(1))
pivot_crop_country.to_csv("detection_rate_per_crop_country.csv")

# Per country x year
rate_by_country_year = detection_rate_table(sample_df, ["samplecountry", "harvest_year"])
pivot_country_year = rate_by_country_year.pivot(
    index="samplecountry", columns="harvest_year", values="detection_rate_pct"
)
print("\nDetection rate per country x year (%):")
print(pivot_country_year.round(1))
pivot_country_year.to_csv("detection_rate_per_country_year.csv")


Detection rate per crop:
product_harmonised  n_samples  n_detected  detection_rate_pct
            barley        694         514               74.06
             maize        287         124               43.21
             wheat        755         448               59.34

Detection rate per year:
 harvest_year  n_samples  n_detected  detection_rate_pct
       2019.0        256         145               56.64
       2020.0        368         237               64.40
       2021.0        382         282               73.82
       2022.0        426         252               59.15
       2023.0        304         170               55.92

Detection rate per country:
samplecountry  n_samples  n_detected  detection_rate_pct
           cz        248         190               76.61
           de        224         136               60.71
           dk        840         609               72.50
           nl        424         151               35.61

Detection rate per crop x year (%):
harvest